# **GPL / GenQ Experiments**

1. **Generate training data** using `run_generation` (from `gpl_generate.py`).
2. **Train student models** using `run_training` (from `gpl_train.py`).

Each experiment has a unique name (*tag*) and stores its results in separate directories:

- **Data:** `runs/data_generated/generated_<tag>/dense/<dataset>/` and `runs/generated_<tag>/bm25/<dataset>/`
- **Models:** `runs/Models/models_<tag>/<method>_<student>/<dataset>/`

Existing training data and trained models are reused rather than regenerated or retrained, allowing the notebook to be rerun without losing previous work.

In [ ]:
import sys
from pathlib import Path
import gpl


PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from gpl_generate import run_generation
from gpl_train import run_training, ROOT

## 1. Configuration


In [ ]:
# Original Datasets (BEIR format)
DATASETS = {
    "scifact": PROJECT_ROOT / "data" /  "scifact",
    "nfcorpus": PROJECT_ROOT / "data" / "nfcorpus",
    "arguana": PROJECT_ROOT / "data" /  "arguana"
}

# Resized corpus size, queries per passage, and GPL training steps for each dataset.
# gpl_steps = new_size * queries_per_passage / 32  ->  one pass over all generated queries
CONFIGS = {
    "scifact": {"new_size": 2592, "queries_per_passage": 3, "gpl_steps": 243, "neg_per_query": 50},
    "nfcorpus": {"new_size": 1816, "queries_per_passage": 4, "gpl_steps": 227, "neg_per_query": 50},
    "arguana": {"new_size": 2892, "queries_per_passage": 3, "gpl_steps": 271, "neg_per_query": 50},
}

# Students: models to be trained
STUDENTS = {
    "BGE": "BAAI/bge-small-en-v1.5",
    "MiniLM": "sentence-transformers/all-MiniLM-L6-v2",
    "GPL_distilbert":"GPL/msmarco-distilbert-margin-mse"}

## **2. Data Generation**

If you modify any generation settings, use a **new tag** to avoid mixing data from different experiments.

To reuse the data previously generated with `generate.py`, move the directories:

- `runs/generated_gpl_smoke/test_<New_tag>/<dataset>` → `runs/data_generated/generated_<New_tag>/dense/<dataset>`
- `runs/generated_gpl_smoke/test_bm25_<New_tag>/<dataset>` → `runs/data_generated/generated_<New_tag>/bm25/<dataset>`

In [ ]:
# Base data generation experiment:
run_generation(DATASETS, CONFIGS, tag="base")

## **3. Training**

Each experiment corresponds to one entry in the list:

- `model_tag`: experiment name (directory `runs/models_<model_tag>/`).
- `gen_tag`: specifies which generated data are used.
- `lr`, `scheduler`, and `rescale_range`: hyperparameters varied across experiments.

In [ ]:
TRAINING_EXPERIMENTS = [
    {"model_tag": "base", "gen_tag": "base", "lr": 2e-5, "scheduler": "constantlr", "rescale_range": (-2.0, 2.0)},
    # EX 2: 
    # {"model_tag": "base", "gen_tag": "Neg_query_10", "lr": 2e-5, "scheduler": "constantlr", "rescale_range": (-2.0, 2.0)},
    
    #EX 3:
    #{"model_tag": "base", "gen_tag": "Step_x2", "lr": 2e-5, "scheduler": "constantlr", "rescale_range": (-2.0, 2.0)}
    ]

for exp in TRAINING_EXPERIMENTS:
    run_training(
        datasets=DATASETS,
        students=STUDENTS,
        gen_tag=exp["gen_tag"],
        model_tag=exp["model_tag"],
        lr=exp["lr"],
        scheduler=exp["scheduler"],
        rescale_range=exp["rescale_range"],
    )